# Backtest: парная стратегия LKOH / ROSN

Спред = log(P_LKOH / P_ROSN) в 18:00. Сигнал -1, если спред > mean + 3σ; +1, если спред < mean − 3σ; иначе 0 (позиции закрываются). mean и σ — по предыдущим 60 дням.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from timer import Timer
from instrument_info import InstrumentInfo
from accounting import AccountingController
from datafeeder import HistoricalDataFeeder
from orderprocessor import SimOrderProcessor
from performance_reporter import PerformanceReporter
from strategies.generic import CapitalHandler
from strategies.pair import StrategyPair
from environment import Environment

## Сборка окружения

In [ ]:
RAW = 'C:/Users/alexl/Dropbox/Projects/NeuroHedgeFund/historical data/raw'
STRATEGY_ID = 'pair_LKOH_ROSN'

timer = Timer()
lkoh, rosn = InstrumentInfo('LKOH', 'LKOH', 'LKOH'), InstrumentInfo('ROSN', 'ROSN', 'ROSN')
datafeeder = HistoricalDataFeeder(timer, RAW, [lkoh, rosn], at='18:00:00')

accountingController = AccountingController(timer)
acc_lkoh, acc_rosn = accountingController.create_accounting(lkoh, STRATEGY_ID), accountingController.create_accounting(rosn, STRATEGY_ID)

capitalHandler = CapitalHandler()
capitalHandler.set_capital(STRATEGY_ID, 1_000_000)

orderProcessor = SimOrderProcessor(accountingController, datafeeder)
reporter = PerformanceReporter(datafeeder, accountingController, capitalHandler)
strategy = StrategyPair(STRATEGY_ID, datafeeder, orderProcessor, capitalHandler, acc_lkoh, acc_rosn, window=60, k=2)

env = Environment(timer, datafeeder, accountingController, orderProcessor, strategy, reporter)

## Запуск

Окружение прогоняется один раз. Для повторного запуска заново выполнить ячейку сборки.

In [ ]:
metrics = env.run()

def fmt(name, v):
    if name.endswith('%'):
        return f'{v:.2f} %'
    if name.endswith('руб.'):
        return f'{v:,.0f}'.replace(',', ' ')
    return f'{v:.3f}'

pd.Series({k: fmt(k, v) for k, v in metrics.items()}, name='Значение').to_frame()

In [ ]:
history = pd.DataFrame(reporter.history, columns=['time', 'pl', 'turnover']).set_index('time')
history.pl.plot(title='PL нарастающим итогом, руб.', figsize=(12, 4), grid=True);